In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

In [9]:
df = yf.download("TSLA", start="2018-01-01", end="2024-01-01")

[*********************100%***********************]  1 of 1 completed


In [22]:
df.isnull().sum()

Price   Ticker
Close   TSLA      0
Return            0
MA5               0
MA10              0
Target            0
dtype: int64

In [10]:
df.head(5)

Price,Close,High,Low,Open,Volume
Ticker,TSLA,TSLA,TSLA,TSLA,TSLA
Date,,,,,
2018-01-02,21.368668,21.474001,20.733334,20.799999,65283000
2018-01-03,21.150000,21.683332,21.036667,21.400000,67822500
2018-01-04,20.974667,21.236668,20.378668,20.858000,149194500
2018-01-05,21.105333,21.149332,20.799999,21.108000,68868000
2018-01-08,22.427334,22.468000,21.033333,21.066668,147891000


In [11]:
df = df[['Close']]
df.dropna(inplace=True)

In [13]:
df['Return'] = df['Close'].pct_change()
df['MA5'] = df['Close'].rolling(5).mean()
df['MA10'] = df['Close'].rolling(10).mean()

df.dropna(inplace=True)

In [14]:
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df.dropna(inplace=True)

In [15]:
X = df[['Return', 'MA5', 'MA10']]
y = df['Target']

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

tscv = TimeSeriesSplit(n_splits=5)

scores = []

for train_index, test_index in tscv.split(X):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = XGBClassifier(n_estimators=100,max_depth=5,learning_rate=0.1,random_state=42,eval_metric='logloss')
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    score = accuracy_score(y_test, y_pred)
    scores.append(score)

print("Scores:", scores)
print("Average Accuracy:", np.mean(scores))

Scores: [0.472, 0.472, 0.548, 0.5, 0.528]
Average Accuracy: 0.504
